# Task 2 — Wheat flours: source download and benchmark preparation

## Goal

Download the spectra and protein references from ChemProject, combine them by source row, and recreate the fixed 80/20 benchmark split with `random_state=123`. The source sample label is excluded from the model variables; the retained integer index records the source row.

The original download is held in a temporary directory and removed at the end. Only the generated benchmark CSV files remain in the task directory.

## Original source

- Dataset page: <https://www.chemproject.org/chemdata>
- Spectra: <https://www.chemproject.org/media/data/x_140farines>
- References: <https://www.chemproject.org/media/data/y_140farines>

## Setup

Set the task parameters, locate the output directory, and define download and verification helpers. The comparison runs before the generated files replace any existing local copies.

In [1]:
TASK_FOLDER = "2-Wheat_flours_protein"
INCLUDE_INDEX = True
COMPARISON_ATOL = 1e-12
SPECTRA_URL = "https://www.chemproject.org/media/data/x_140farines"
REFERENCE_URL = "https://www.chemproject.org/media/data/y_140farines"

In [2]:
from pathlib import Path
from urllib.request import Request, urlopen
import hashlib
import io
import shutil
import tempfile
import zipfile

import numpy as np
import pandas as pd
from scipy.io import loadmat


def find_task_dir(folder_name):
    """Support running from either the task directory or repository root."""
    candidates = [Path.cwd(), Path.cwd() / "datasets" / folder_name]
    for candidate in candidates:
        if candidate.name == folder_name and (candidate / "download_data.md").exists():
            return candidate.resolve()
    raise FileNotFoundError(
        f"Run this notebook from {folder_name!r} or from the repository root."
    )


def download(url, destination):
    """Download a source file without retaining it in the repository."""
    request = Request(url, headers={"User-Agent": "NIRBENCH-DL/0.1 dataset preparation"})
    with urlopen(request, timeout=120) as response, destination.open("wb") as output:
        shutil.copyfileobj(response, output)
    print(f"Downloaded {destination.name} ({destination.stat().st_size:,} bytes)")


def csv_roundtrip(frame, include_index):
    """Represent a generated frame as it will appear after CSV serialization."""
    buffer = io.StringIO()
    frame.to_csv(buffer, index=include_index, lineterminator="\r\n")
    buffer.seek(0)
    return pd.read_csv(buffer)


def verify_and_write(train_frame, test_frame, include_index, atol):
    """Compare with published partitions when present, then write both CSV files."""
    results = []
    for split, generated in (("train", train_frame), ("test", test_frame)):
        output_path = TASK_DIR / f"data_{split}.csv"
        serialized = csv_roundtrip(generated, include_index)

        if output_path.exists():
            published = pd.read_csv(output_path)
            pd.testing.assert_frame_equal(
                serialized,
                published,
                check_dtype=False,
                check_exact=False,
                rtol=0,
                atol=atol,
            )
            max_delta = np.max(
                np.abs(
                    serialized.select_dtypes(include="number").to_numpy(dtype=float)
                    - published.select_dtypes(include="number").to_numpy(dtype=float)
                )
            )
            print(
                f"{split}: matches the published partition "
                f"(shape={serialized.shape}, maximum absolute difference={max_delta:.3g})"
            )
        else:
            print(f"{split}: no published CSV was present; generated shape={serialized.shape}")

        generated.to_csv(
            output_path,
            index=include_index,
            lineterminator="\r\n",
        )
        digest = hashlib.sha256(output_path.read_bytes()).hexdigest()
        results.append({
            "partition": split,
            "rows": len(generated),
            "spectral_variables": generated.shape[1] - 1,
            "sha256": digest,
        })

    return pd.DataFrame(results)


TASK_DIR = find_task_dir(TASK_FOLDER)
temporary_download = tempfile.TemporaryDirectory(prefix=f"nirbench_{TASK_FOLDER}_")
SOURCE_DIR = Path(temporary_download.name)
print(f"Preparing task {TASK_FOLDER}")

Preparing task 2-Wheat_flours_protein


## Download and prepare

Download the authoritative source and apply the task specific conversion recorded in `DATA_LICENSES.md`.

In [3]:
from sklearn.model_selection import train_test_split

spectra_path = SOURCE_DIR / "x_140farines.csv"
reference_path = SOURCE_DIR / "y_140farines.csv"
download(SPECTRA_URL, spectra_path)
download(REFERENCE_URL, reference_path)

spectra = pd.read_csv(spectra_path, sep=";", index_col=0)
references = pd.read_csv(reference_path, sep=";", index_col=0)
assert len(spectra) == len(references) == 140

all_data = spectra.reset_index(drop=True)
all_data = all_data.assign(PROTREF=references.reset_index(drop=True)["PROTREF"])
train_data, test_data = train_test_split(
    all_data,
    test_size=0.20,
    random_state=123,
)

assert train_data.shape == (112, 526)
assert test_data.shape == (28, 526)
print("Combined 525 spectral variables with PROTREF and applied the fixed split.")

Downloaded x_140farines.csv (509,742 bytes)


Downloaded y_140farines.csv (1,520 bytes)
Combined 525 spectral variables with PROTREF and applied the fixed split.


/tmp/ipykernel_23541/1092611686.py:13: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  all_data = all_data.assign(PROTREF=references.reset_index(drop=True)["PROTREF"])


## Verify and write

When published CSVs are present, compare their shapes, columns, row order, indices, and numerical values with the generated partitions. Then write `data_train.csv` and `data_test.csv` using deterministic CRLF line endings.

In [4]:
summary = verify_and_write(
    train_data,
    test_data,
    include_index=INCLUDE_INDEX,
    atol=COMPARISON_ATOL,
)
display(summary)
temporary_download.cleanup()
print("Temporary source files removed. Preparation complete.")

train: matches the published partition (shape=(112, 527), maximum absolute difference=0)
test: matches the published partition (shape=(28, 527), maximum absolute difference=0)


,partition,rows,spectral_variables,sha256
0,train,112,525,c4a80a15d2dbe3da6af0cdb40ac7c48185b7d145e07b0f...
1,test,28,525,1eb9e4fb3f2e6a32c4cfd4714cc1edde75b9ab6e1abbd0...


Temporary source files removed. Preparation complete.
